In [1]:
import os

# =========================
# FIX OPENMP + CUDA MEMORY
# =========================
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from config import ROOT

from src.data.STL_dataset import SLTDataset
from src.models.SLT_model import SLTModel
from src.utils.vocabulary import Vocabulary
import torch
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = rf"{ROOT}\datasets\processed\features"
CSV_PATH = rf"{ROOT}\datasets\annotations\how2sign_train.csv"
SAVE_DIR = rf"{ROOT}\models"
BATCH_SIZE = 2
EPOCHS = 10
LR = 1e-4

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2

In [2]:
def align(left, right):
    T = min(left.shape[0], right.shape[0])

    left = left[:, :T]
    right = right[:, :T]

    return left, right


def collate_fn(batch):

    lefts = [x[0] for x in batch]
    rights = [x[1] for x in batch]
    texts = [x[2] for x in batch]

    lefts = pad_sequence(
        lefts,
        batch_first=True
    )

    rights = pad_sequence(
        rights,
        batch_first=True
    )

    texts = pad_sequence(
        texts,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return lefts, rights, texts

df = pd.read_csv(CSV_PATH, sep="\t")

vocab = Vocabulary()

for s in tqdm(df["SENTENCE"].tolist()):

    vocab.build_vocab(
        s.lower().split()
    )
dataset = SLTDataset(DATA_PATH)


100%|██████████| 31165/31165 [00:54<00:00, 568.81it/s]


In [3]:

# Sample
left, right, gt_text = dataset[0]

T = min(left.shape[0], right.shape[0])

left = left[:T, :]
right = right[:T, :]

print(f"Left: {left.shape}")
print(f"Right: {right.shape}")
print(f"Text: {gt_text}")


left = left.unsqueeze(0).to(DEVICE)
right = right.unsqueeze(0).to(DEVICE)

Left: torch.Size([364, 512])
Right: torch.Size([364, 512])
Text: tensor([ 1,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 15, 18,  4,
        19, 20, 21,  2])


D:\SignDetection\src\data\STL_dataset.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  text = torch.load(item["text"])
C:\Users\2507\AppData\Local\Temp\ipykernel_3704\17

In [4]:
MODEL_PATH = rf"{ROOT}\models\best_model.pt"
model = SLTModel(
    vocab_size=len(vocab.word2idx)
).to(DEVICE)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

C:\Users\2507\AppData\Local\Temp\ipykernel_3704\3232682788.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)


SLTModel(
  (fusion): Linear(in_features=1024, out_features=512, bias=True)
  (encoder): TemporalEncoder(
    (position): PositionalEncoding()
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (decoder): TextDecoder(
    (embedding): Embedding(26976, 512, padding_idx=0)
    (position): PositionalEncoding

In [6]:

tokens = [SOS_IDX]

for _ in range(20):

    inp = torch.tensor(tokens).unsqueeze(0).to(DEVICE)

    out, _ = model(left, right, inp)

    next_token = out[:, -1].argmax(-1).item()

    tokens.append(next_token)

    if next_token == EOS_IDX:
        break

pred_sentence = vocab.decode(tokens)
gt_sentence = vocab.decode(gt_text.tolist())
print(len(str.split(pred_sentence, sep=" ")))
print("PRED:", pred_sentence)
print("GT  :", gt_sentence)

20
PRED: and then you can go ahead and take a little bit of it, and just go ahead and just kind
GT  : and i call them decorative elements because basically all they're meant to do is to enrich and color the page.
